In [0]:
%sql
select * from lh_nautical.gold_lh_nautical.fatos_vendas
          where id_product = '54' and sale_date>'2023-12-31'

In [0]:
#treino = spark.sql("""
          #select * from lh_nautical.gold_lh_nautical.fatos_vendas
          #where id_product = '54' and sale_date between '2019-01-01' and '2023-12-31'
          #""")

#teste = spark.sql("""
          #select * from lh_nautical.gold_lh_nautical.fatos_vendas
          #where id_product = '54' and sale_date>'2023-12-31'
         # """)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta


# FILTRO DO PRODUTO
treino_filtrado = treino.filter(F.col("id_product") == 54)
teste_filtrado = teste.filter(F.col("id_product") == 54)


#  AGREGAÇÃO DIÁRIA
treino_diario = (
    treino_filtrado
    .groupBy("sale_date")
    .agg(F.sum("qtd").alias("quantidade_diaria"))
    .orderBy("sale_date")
)

teste_diario = (
    teste_filtrado
    .groupBy("sale_date")
    .agg(F.sum("qtd").alias("quantidade_diaria"))
    .orderBy("sale_date")
)


# MÉDIA MÓVEL (BASELINE)
janela_7_dias = (
    Window
    .orderBy("sale_date")
    .rowsBetween(-7, -1)
)
treino_com_media = (
    treino_diario
    .withColumn("media_movel_7d", F.avg("quantidade_diaria").over(janela_7_dias))
)

# PREVISÃO RECURSIVA (JANEIRO/2024)
ultimos_7_dias = (
    treino_diario
    .orderBy("sale_date", ascending=False)
    .limit(7)
    .orderBy("sale_date")
)

historico = [linha["quantidade_diaria"] for linha in ultimos_7_dias.collect()]

data_inicial = datetime(2024, 1, 1)
datas_janeiro = [data_inicial + timedelta(days=i) for i in range(31)]

previsoes = []

for data in datas_janeiro:
    previsao = sum(historico[-7:]) / 7
    previsoes.append((data, float(previsao)))
    historico.append(previsao)

df_previsao = spark.createDataFrame(previsoes, ["sale_date", "previsao"])


# AVALIAÇÃO (MAE)
df_avaliacao = (
    df_previsao
    .join(teste_diario, on="sale_date", how="inner")
    .withColumn("erro_absoluto", F.abs(F.col("quantidade_diaria") - F.col("previsao")))
)

mae = df_avaliacao.agg(F.avg("erro_absoluto")).collect()[0][0]

print(f"MAE: {mae}")


#  OUTPUTS
display(treino_com_media)
display(df_previsao)
display(df_avaliacao)

In [0]:
from pyspark.sql import functions as F

df_primeira_semana = df_previsao.filter(
    (F.col("sale_date") >= "2024-01-01") &
    (F.col("sale_date") <= "2024-01-07"))
df_primeira_semana = df_primeira_semana.withColumn(
    "previsao_arredondada",
    F.round("previsao", 0)
)
soma_total = (
    df_primeira_semana
    .agg(F.sum("previsao_arredondada"))
    .collect()[0][0]
)

print(f"Soma da previsão (1ª semana): {soma_total}")
display(df_primeira_semana)